# The Brachistochrone — the Fastest Slide as a Convex Problem

**Continuous Optimization (MasterMath) — Lecture 1 demo.**

Johann Bernoulli's challenge (1696): along which curve does a bead slide under gravity, without friction, from the origin to a point $L$ to the right and $H$ below, in the *least time*? By conservation of energy the speed at depth $y$ is $\sqrt{2gy}$, whatever the curve.

**The convexity trick.** Parametrise the curve by its *depth*: horizontal position $x(y)$ for $y \in [0, H]$. The travel time becomes

$$T[x] \;=\; \int_0^H \frac{\sqrt{1 + x'(y)^2}}{\sqrt{2gy}}\, dy, \qquad x(0)=0,\; x(H)=L.$$

The integrand depends on $x$ only through $x'(y)$ — a linear operation — and $p \mapsto \sqrt{1+p^2}$ is convex, scaled by a positive weight for each $y$. Hence $T$ is a **convex** functional with linear boundary conditions: a convex optimization problem! (This parametrisation assumes the curve is a graph over depth, which holds for the optimal curve whenever $L/H \le \pi/2$.)

**Discretisation.** On a depth grid $y_k = kh$ with positions $x_k$, the time along a straight segment integrates in closed form:

$$t_k = \sqrt{h^2 + (x_{k+1}-x_k)^2}\cdot \frac{1}{h}\sqrt{\tfrac{2}{g}}\left(\sqrt{y_{k+1}} - \sqrt{y_k}\right),$$

so the objective $\sum_k t_k$ is the *exact* travel time of the piecewise-linear path — a norm of an affine expression per segment, i.e. a small second-order-cone program.

In [ ]:
# Colab does not ship CVXPY by default; install it (skipped if already present).
try:
    import cvxpy  # noqa: F401
except ImportError:
    %pip install -q cvxpy

In [ ]:
import numpy as np
import cvxpy as cp

g = 9.81         # gravity (m/s^2)
L, H = 1.0, 1.0  # end point: L to the right, H straight down (keep L/H <= pi/2)

# Discretise the depth interval [0, H] into N segments.
N = 500
h = H / N
y = np.arange(N + 1) * h
w = np.sqrt(2 / g) * (np.sqrt(y[1:]) - np.sqrt(y[:-1])) / h

# Decision variable: horizontal position x_k at depth y_k.
x = cp.Variable(N + 1)

# Segment length sqrt(h^2 + (x_{k+1}-x_k)^2): the norm of an affine expression.
seg = cp.norm(cp.vstack([np.full(N, h), cp.diff(x)]), 2, axis=0)

objective = cp.Minimize(w @ seg)   # exact travel time of the piecewise-linear path
constraints = [x[0] == 0, x[N] == L]

problem = cp.Problem(objective, constraints)
problem.solve()

print(f"Status: {problem.status}")
print(f"Travel time (convex optimization) : {problem.value:.6f} s")

**The classical answer** (Bernoulli, Newton, Leibniz, l'Hôpital, and Jacob Bernoulli all solved it) is a **cycloid**: $x = r(\theta - \sin\theta)$, $y = r(1 - \cos\theta)$, the path traced by a point on a rolling wheel, with travel time $\Theta\sqrt{r/g}$ at the angle $\Theta$ where it reaches $(L, H)$. Let us check that the solver rediscovers it — the optimal time should approach the cycloid's from above as $N$ grows, since piecewise-linear paths are slightly suboptimal.

In [ ]:
from scipy.optimize import brentq

# Angle at which the cycloid through the origin reaches (L, H).
theta = brentq(lambda t: (t - np.sin(t)) / (1 - np.cos(t)) - L / H, 1e-3, np.pi)
r = H / (1 - np.cos(theta))
print(f"Travel time (Bernoulli's cycloid) : {theta * np.sqrt(r / g):.6f} s")

# A straight slide, for comparison (the same weights time it exactly).
straight = np.hypot(h, L / N) * np.sum(w)
print(f"Travel time (straight line)       : {straight:.6f} s")

In [ ]:
import matplotlib.pyplot as plt

t_c = np.linspace(0, theta, 400)

plt.figure(figsize=(7, 5))
plt.plot(x.value, y, lw=4, alpha=0.5, label="convex optimization")
plt.plot(r * (t_c - np.sin(t_c)), r * (1 - np.cos(t_c)), "k--", lw=1.5,
         label="Bernoulli's cycloid")
plt.plot([0, L], [0, H], ":", color="tab:red", label="straight line")
plt.scatter([0, L], [0, H], color="k", zorder=3)
plt.gca().invert_yaxis()               # depth increases downward
plt.xlabel("horizontal position $x$")
plt.ylabel("depth $y$")
plt.title("The fastest slide is a cycloid")
plt.legend()
plt.show()

**Things to try:** increase $N$ and watch the optimal time converge to the cycloid's; move the end point (what happens to the parametrisation when $L/H > \pi/2$?); add an obstacle the slide must avoid, e.g. the linear constraints $x_k \ge x^{\mathrm{obs}}_k$ — still convex!